# Exploratory analysis of renewable electricity generation

Run the setup cell first, then follow the lesson in order.

In [ ]:
import importlib.util
import subprocess
import sys

required = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.2,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

from hashlib import sha256
from pathlib import Path
from urllib.request import urlopen

data_source = 'https://raw.githubusercontent.com/roderickperez/EAGE_PythonRenewableEnergyCourse/main/EAGE_PythonRenewableEnergyCourse/'
datasets = {'data/examples/eurostat_at_wind_2023.csv': '52894468c107f540620a7d7cf2de327e43f56d2fcc3999bc5cb4bca3cc7ac111'}
for relative_path, expected in datasets.items():
    target = Path(relative_path)
    if target.is_file() and sha256(target.read_bytes()).hexdigest() == expected:
        continue
    with urlopen(data_source + relative_path, timeout=30) as response:
        payload = response.read()
    if sha256(payload).hexdigest() != expected:
        raise ValueError(f"Course dataset checksum mismatch: {relative_path}")
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".download")
    temporary.write_bytes(payload)
    temporary.replace(target)

Use Eurostat’s monthly net wind electricity generation for Austria in 2023. Each row measures generation over a calendar month in GWh. A longer month has more time to accumulate energy. Load the twelve values from the [bundled Eurostat extract](https://roderickperez.github.io/EAGE_PythonRenewableEnergyCourse/sandbox/generated/assets/data/examples/eurostat_at_wind_2023.csv).

## Descriptive statistics and time series

Check the monthly index and describe the distribution. Convert GWh to MWh and divide by calendar hours to estimate monthly average power.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

asset = Path("data/examples/eurostat_at_wind_2023.csv")
csv_path = next(candidate / asset for parent in (Path.cwd(), *Path.cwd().parents)
                for candidate in (parent, parent / "EAGE_PythonRenewableEnergyCourse")
                if (candidate / asset).is_file())
observations = pd.read_csv(csv_path)
assert observations["geo"].eq("AT").all() and observations["source"].eq("RA300").all()
assert observations["unit"].eq("GWH").all() and observations["freq"].eq("M").all()
observations["month"] = pd.to_datetime(observations["month"], format="%Y-%m", utc=True)
energy = observations.set_index("month")["generation_gwh"].sort_index()
months = pd.date_range("2023-01-01", periods=12, freq="MS", tz="UTC")
assert energy.index.is_unique and energy.index.equals(months)
assert energy.notna().all() and energy.ge(0).all()
print(energy.describe().round(2))
hours = pd.Series(months.days_in_month * 24, index=months)
mean_power = (1_000 * energy / hours).rename("mean_power_mw")
assert np.isclose((mean_power * hours).sum(), 1_000 * energy.sum())

## Seasonal variation

Matplotlib preserves chronological order. Seaborn compares monthly averages by calendar season. The DJF group contains January, February and December within the sample year. Several years are needed to establish a recurring seasonal pattern.

In [ ]:
season_order = ["DJF", "MAM", "JJA", "SON"]
season_by_month = dict(zip(range(1, 13), [
    "DJF", "DJF", "MAM", "MAM", "MAM", "JJA",
    "JJA", "JJA", "SON", "SON", "SON", "DJF",
]))
data = mean_power.to_frame()
data["season"] = data.index.month.map(season_by_month)

fig, (series_ax, season_ax) = plt.subplots(2, 1, figsize=(8, 6), layout="constrained")
series_ax.plot(energy.index, energy, marker="o")
series_ax.set(ylabel="Generation (GWh)", title="Austria monthly wind generation, 2023 (UTC)")
sns.stripplot(data=data, x="season", y="mean_power_mw", order=season_order,
              jitter=False, ax=season_ax)
season_ax.set(xlabel="Calendar-season group", ylabel="Monthly average power (MW)")
plt.show()

## Practice

Find the season with the lowest average power weighted by hours. Set July’s generation to missing and report the share of summer hours covered and the subtotal from the remaining observations.

### Worked solution

In [ ]:
season = data["season"]
weighted_mw = 1_000 * energy.groupby(season).sum() / hours.groupby(season).sum()
assert weighted_mw.idxmin() == "JJA"

with_gap = energy.copy()
with_gap.loc["2023-07-01"] = np.nan
summer = season.eq("JJA")
reported = summer & with_gap.notna()
coverage = hours[reported].sum() / hours[summer].sum()
assert np.isclose(coverage, 61 / 92)
print(f"Summer coverage: {coverage:.1%}")
print(f"Reported summer subtotal: {with_gap[summer].sum(min_count=1):.2f} GWh")

[Eurostat dataset](https://ec.europa.eu/eurostat/databrowser/view/nrg_cb_pem/default/table?lang=en) · [Longer time-series case study](https://roderickperez.github.io/EAGE_PythonRenewableEnergyCourse/sandbox/generated/assets/section4/timeSeriesEnergyConsumption.ipynb)